# Run the Support Ticket Triage app from Colab

Use this notebook if your own computer has no GPU. It starts the Streamlit app on a free Colab GPU and gives you a public link you can open in your browser (and take screenshots of).

**Before you start:** Runtime > Change runtime type > pick a GPU (T4 is enough).

You need two things: this repository's code, and the `final_model` folder produced by the Milestone 3 notebook (the fine-tuned weights, about 1 GB).

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/support-ticket-triage.git"   # <-- change this to your repository

import os
if not os.path.exists("/content/app"):
    !git clone {REPO_URL} /content/app
%cd /content/app
!ls

## 2. Get the fine-tuned model

Pick **one** of these.

**A. The weights are on the Hugging Face Hub** (recommended; works from any machine). Set `MODEL = "your-username/repo-name"` below. For a private repo, add a Hugging Face token to Colab's Secrets (key icon on the left) under the name `HF_TOKEN`.

**B. The weights are in your Google Drive** as a `final_model` folder. Leave `MODEL = None` and the Drive cell below copies it.

**C. You still have the training session open** and the folder `/content/m3_run/final_model` exists. Leave `MODEL = None`; the cell finds it.

In [ ]:
import os
MODEL = None   # option A: e.g. "your-username/support-triage-qwen0.5b"

if MODEL is None:
    if os.path.exists("/content/m3_run/final_model"):           # option C
        MODEL = "/content/m3_run/final_model"
    else:                                                       # option B: copy from Drive
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_PATH = "/content/drive/MyDrive/final_model"       # <-- change if you saved it elsewhere
        !cp -r "{DRIVE_PATH}" /content/app/final_model
        MODEL = "/content/app/final_model"
os.environ["TRIAGE_MODEL"] = MODEL
print("model:", MODEL)

## 3. Install and verify the model

`check_model.py` sends 10 probe messages and checks that the loaded weights behave like the fine-tuned model (valid JSON tickets), not the base model. Run it before you take screenshots.

In [ ]:
!pip install -q streamlit
!python check_model.py

## 4. Start the app and open the public link

Wait for the line that contains `trycloudflare.com`, then click it. Keep this cell running while you use the app; interrupt the cell to stop the app.

In [ ]:
import subprocess, time, re, os

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

env = dict(os.environ, TRIAGE_MODEL=MODEL)
st_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)
time.sleep(8)
tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("Open the app here:", m.group(0))
        break
while True:          # keep the cell alive so the app stays up
    time.sleep(60)

## Optional: save the model so you never have to retrain

If you are still in the training session, copy `final_model` to Drive or push it to the Hugging Face Hub.

In [ ]:
# to Google Drive
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r /content/m3_run/final_model /content/drive/MyDrive/final_model

# to the Hugging Face Hub (private repo)
# from huggingface_hub import login; login()
# from transformers import AutoModelForCausalLM, AutoTokenizer
# P = "/content/m3_run/final_model"
# AutoModelForCausalLM.from_pretrained(P).push_to_hub("your-username/support-triage-qwen0.5b", private=True)
# AutoTokenizer.from_pretrained(P).push_to_hub("your-username/support-triage-qwen0.5b", private=True)